# Q6: What are the key steps involved in the Face Recognition process?
**Topic:** Computer-vision | **Difficulty:** Medium | **Time:** 8 min
**Sheet:** Grind75ML

---

## Question
What are the key steps involved in the Face Recognition process?

## Detailed Answer

Face recognition is a multi-stage pipeline:

### Stage 1: Face Detection
Locate faces in the image with bounding boxes.

**Common methods:**
- **MTCNN** (Multi-task Cascaded CNN): 3-stage cascade (P-Net → R-Net → O-Net), detects faces + landmarks
- **RetinaFace**: Single-stage detector with multi-task learning (face box + 5 landmarks + 3D face)
- **BlazeFace**: Lightweight detector optimized for mobile (used in MediaPipe)
- **YOLO-Face**: YOLO adapted for face detection

### Stage 2: Face Alignment
Normalize face orientation using detected landmarks (eyes, nose, mouth corners).

**Process:**
1. Detect 5 facial landmarks (2 eyes, nose tip, 2 mouth corners)
2. Compute affine transformation to align to a canonical face template
3. Apply transformation to crop and align the face

**Why:** Reduces variation due to pose, making embedding comparison more reliable.

### Stage 3: Feature Extraction (Embedding)
Convert the aligned face to a compact vector representation (embedding).

**Key models:**
- **FaceNet** (Google): Triplet loss, 128-d embedding. $L = \sum [\|f(a)-f(p)\|^2 - \|f(a)-f(n)\|^2 + \alpha]_+$
- **ArcFace**: Additive Angular Margin Loss — pushes same-class embeddings closer on the hypersphere
- **CosFace**: Additive cosine margin loss
- **SphereFace**: Multiplicative angular margin

**Embedding dimension:** Typically 128 or 512 floats

### Stage 4: Face Matching / Verification
Compare embeddings to determine identity.

**Two modes:**
- **Verification (1:1)**: Is this person who they claim to be? Compare two embeddings with a threshold
  - Distance metric: Cosine similarity or L2 distance
  - $\text{match} = \text{distance}(e_1, e_2) < \tau$
- **Identification (1:N)**: Who is this person? Find nearest embedding in a database
  - Use approximate nearest neighbor search (FAISS, Annoy)

### Stage 5 (Optional): Liveness Detection / Anti-Spoofing
Prevent spoofing with printed photos, screen replays, or 3D masks.

**Methods:**
- Texture analysis (LBP patterns differ for printed photos)
- Depth estimation (3D vs 2D)
- Challenge-response (blink, turn head)
- Infrared imaging

### Complete Pipeline:
```
Image → Face Detection → Alignment → Feature Extraction → Matching
         (MTCNN)        (Affine)     (ArcFace/FaceNet)   (Cosine/L2)
                                                          ↕
                                                    Face Database
                                                     (FAISS)
```

In [ ]:
import numpy as np


def cosine_similarity(embedding1: np.ndarray, embedding2: np.ndarray) -> float:
    """Compute cosine similarity between two face embeddings."""
    dot_product = np.dot(embedding1, embedding2)
    norm1 = np.linalg.norm(embedding1)
    norm2 = np.linalg.norm(embedding2)
    return dot_product / (norm1 * norm2)


def l2_distance(embedding1: np.ndarray, embedding2: np.ndarray) -> float:
    """Compute L2 (Euclidean) distance between embeddings."""
    return np.linalg.norm(embedding1 - embedding2)


def verify_face(emb1: np.ndarray, emb2: np.ndarray, 
                threshold: float = 0.6, metric: str = 'cosine') -> tuple[bool, float]:
    """Verify if two face embeddings belong to the same person.
    
    Returns:
        (is_same_person, similarity_score)
    """
    if metric == 'cosine':
        score = cosine_similarity(emb1, emb2)
        return score > threshold, score
    else:  # L2
        dist = l2_distance(emb1, emb2)
        return dist < threshold, dist


def identify_face(query_emb: np.ndarray, database: dict[str, np.ndarray],
                  threshold: float = 0.6) -> tuple[str, float]:
    """Identify a face by finding the closest match in the database."""
    best_match = 'Unknown'
    best_score = -1.0
    
    for name, db_emb in database.items():
        score = cosine_similarity(query_emb, db_emb)
        if score > best_score:
            best_score = score
            best_match = name
    
    if best_score < threshold:
        return 'Unknown', best_score
    return best_match, best_score


# --- Demo ---
np.random.seed(42)
# Simulate 128-d face embeddings
person_a = np.random.randn(128)
person_a = person_a / np.linalg.norm(person_a)  # normalize

person_a_2 = person_a + np.random.randn(128) * 0.1  # same person, slight variation
person_a_2 = person_a_2 / np.linalg.norm(person_a_2)

person_b = np.random.randn(128)  # different person
person_b = person_b / np.linalg.norm(person_b)

# Verification
is_same, score = verify_face(person_a, person_a_2)
print(f"Same person (A vs A'): match={is_same}, score={score:.4f}")

is_same, score = verify_face(person_a, person_b)
print(f"Diff person (A vs B):  match={is_same}, score={score:.4f}")

# Identification
database = {'Alice': person_a, 'Bob': person_b}
name, score = identify_face(person_a_2, database)
print(f"\nIdentified: {name} (score: {score:.4f})")

## Key Takeaways
- The pipeline is: **Detect → Align → Embed → Match** — each stage matters
- **ArcFace** is currently the SOTA loss function for face embedding training
- **Alignment** is often the most underrated step — it dramatically improves accuracy
- For production: use FAISS for fast nearest-neighbor search in large face databases